In [0]:
#%skip
#This cell doesnt need to be ran and was used as a preliminary step to get the schema correct and debug any issues on initial data load. We will not include this cell in a run.

#We found Input files are messy and have no header, had to start at row 8 

df = spark.read.option('skipRows', 7).csv(
    '/Volumes/jc_demoworkspace/datahive/airline_departures',
    header =True,
    inferSchema =True
)
display(df.orderBy(df['Flight Number'], ascending=True).head(5))

#df.printSchema() 

In [0]:
from pyspark.sql.types import *
#from pyspark.sql.functions import date_format,to_timestamp,to_date
from pyspark.sql import functions as F


In [0]:
schema_new = (
StructType([
    StructField("carrier_code", StringType(), True),
    StructField("date", StringType(), True),
    StructField("flight_number", StringType(), True),
    StructField("tail_number", StringType(), True),
    StructField("destination_airport", StringType(), True),
    StructField("scheduled_departure_time", StringType(), True),
    StructField("actual_departure_time", StringType(), True),
    StructField("scheduled_elapsed_time_minutes", IntegerType(), True),
    StructField("actual_elapsed_time_minutes", IntegerType(), True),
    StructField("departure_delay_minutes", IntegerType(), True),
    StructField("wheelsoff_time", StringType(), True),
    StructField("taxiout_time_minutes", IntegerType(), True),
    StructField("delay_carrier_minutes", IntegerType(), True),
    StructField("delay_weather_minutes", IntegerType(), True),
    StructField("delay_national_aviation_system_minutes", IntegerType(), True),
    StructField("delay_security_minutes", IntegerType(), True),
    StructField("delay_late_aircraft_arrival_minutes", IntegerType(), True)
])
)

departures_df = spark.read.option('skipRows', 8).csv(
    '/Volumes/jc_demoworkspace/datahive/airline_departures',
    schema = schema_new
)
display(departures_df)

In [0]:
#need to get origin airport code from original file

flight_origin  = spark.read.option('skipRows',1).csv(
    '/Volumes/jc_demoworkspace/datahive/airline_departures',
    header =False,
    inferSchema =True
)

flight_origin = flight_origin.withColumn(
    "origin_code",
    F.regexp_extract("_c1", r"\((.*?)\)", 1)
)


origin = flight_origin.select("origin_code").first()[0]
print(origin)

In [0]:
departures_df_enriched = departures_df\
    .withColumn("origin_code",F.lit(origin))\
    .select("*","_metadata.file_name")\
    .withColumn("date",F.to_date("date", "MM/dd/yyyy"))\
    .withColumn("file_uploaded", F.current_timestamp())
#display(departures_df_enriched)

In [0]:
try:
    departures_df.count() == departures_df_enriched.count()
except:
    print("Dataframe count does not match, rows lost during enrichment")

departures_df_enriched.write.option("mergeSchema", "true").mode("overwrite").saveAsTable("jc_demoworkspace.datahive.bronze_airline_departures")
spark